### Test Dataset's Hyperbolicity

The key metric is Gromov's delta-hyperbolicity (δ), which measures how "tree-like" your data is. A value close to 0 means your data has hierarchical structure that hyperbolic embeddings will exploit well.

In [ ]:
import numpy as np
from scipy.spatial import distance_matrix
from tqdm import tqdm
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from rfdetr import RFDETRBase

# --- 1. delta hyperbolicity functions (from hyptorch/delta.py) ---

def delta_hyp(dismat):
    """Computes delta hyperbolicity from a distance matrix."""
    p = 0
    row = dismat[p, :][np.newaxis, :]
    col = dismat[:, p][:, np.newaxis]
    XY_p = 0.5 * (row + col - dismat)
    maxmin = np.max(np.minimum(XY_p[:, :, None], XY_p[None, :, :]), axis=1)
    return np.max(maxmin - XY_p)


def batched_delta_hyp(X, n_tries=10, batch_size=1500):
    """Computes relative delta hyperbolicity with random sampling."""
    vals = []
    for i in tqdm(range(n_tries)):
        idx = np.random.choice(len(X), min(batch_size, len(X)), replace=False)
        X_batch = X[idx]
        distmat = distance_matrix(X_batch, X_batch)
        diam = np.max(distmat)
        if diam > 0:
            delta_rel = 2 * delta_hyp(distmat) / diam
            vals.append(delta_rel)
    return np.mean(vals), np.std(vals)


In [ ]:
# --- 2. Extract decoder features from your trained model ---

model = RFDETRBase(pretrain_weights="output/checkpoint_best_total.pth")
model.model.model.eval()
device = model.model.device

# Load your dataset images
import os, json, cv2
dataset_dir = "Veiculos-Contar-3"  
with open(os.path.join(dataset_dir, "train", "_annotations.coco.json")) as f:
    ann = json.load(f)

print(f"Train images: {len(ann['images'])}")

# Count annotations per class
from collections import Counter
class_counts = Counter(a["category_id"] for a in ann["annotations"])
class_names = {c["id"]: c["name"] for c in ann["categories"]}
for cid, count in sorted(class_counts.items()):
    print(f"  {class_names[cid]:15s}: {count} annotations")

all_features = []
all_labels = []

# Build image_id -> annotations lookup
img_id_to_anns = {}
for a in ann["annotations"]:
    img_id_to_anns.setdefault(a["image_id"], []).append(a)

with torch.no_grad():
    for img_info in tqdm(ann["images"][:500]):
        img_path = os.path.join(dataset_dir, "train", img_info["file_name"])
        pil_img = Image.open(img_path).convert("RGB")

        # Preprocess the same way RF-DETR does
        img_tensor = TF.to_tensor(pil_img).to(device)
        img_tensor = TF.normalize(
            img_tensor,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )
        img_tensor = torch.nn.functional.interpolate(
            img_tensor.unsqueeze(0),
            size=(model.model.resolution, model.model.resolution),
            mode="bilinear",
        )

        # Forward through backbone + transformer decoder (but NOT class_embed)
        from rfdetr.util.misc import nested_tensor_from_tensor_list
        samples = nested_tensor_from_tensor_list(img_tensor)
        features, poss = model.model.model.backbone(samples)
        srcs = [feat.decompose()[0] for feat in features]
        masks = [feat.decompose()[1] for feat in features]

        refpoint = model.model.model.refpoint_embed.weight[:model.model.model.num_queries]
        query_feat = model.model.model.query_feat.weight[:model.model.model.num_queries]

        hs, _, _, _ = model.model.model.transformer(
            srcs, masks, poss, refpoint, query_feat
        )

        # hs shape: [dec_layers, 1, num_queries, hidden_dim]
        # Take last decoder layer, squeeze batch
        decoder_features = hs[-1].squeeze(0)  # [300, 256]

        # Get the class predictions to filter high-confidence detections
        logits = model.model.model.class_embed(hs[-1]).squeeze(0)  # [300, num_classes]
        scores = logits.sigmoid().max(dim=-1)
        conf = scores.values   # [300]
        # Debug: see what confidence scores look like
        if img_info == ann["images"][0]:
            print(f"Max confidence: {conf.max().item():.4f}")
            print(f"Detections > 0.1: {(conf > 0.1).sum().item()}")
            print(f"Detections > 0.3: {(conf > 0.3).sum().item()}")
        classes = scores.indices  # [300]

                # Lower threshold to capture more detections
        keep = conf > 0.1  # lowered from 0.3

        if keep.sum() > 0:
            all_features.append(decoder_features[keep].cpu().numpy())
            all_labels.append(classes[keep].cpu().numpy())

    # After the loop, check what we got
    print(f"Images processed: {len(ann['images'][:500])}")
    print(f"Feature batches collected: {len(all_features)}")

    if len(all_features) == 0:
        raise RuntimeError(
            "No detections found. Try using model.predict() on a single image first "
            "to verify the model is working correctly."
        )

all_features = np.concatenate(all_features, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

Reinitializing detection head with 8 classes


Loading pretrain weights
Train images: 5528
  Bus            : 115 annotations
  Motorcycle     : 400 annotations
  Pickup         : 11049 annotations
  Sedan          : 10840 annotations
  Suv            : 2152 annotations
  Truck          : 5248 annotations
  Van            : 1551 annotations


  0%|          | 1/500 [00:00<03:36,  2.31it/s]

Max confidence: 0.7809
Detections > 0.1: 3
Detections > 0.3: 2


100%|██████████| 500/500 [03:16<00:00,  2.55it/s]

Images processed: 500
Feature batches collected: 500


In [ ]:
# --- 3. Compute delta hyperbolicity ---

mean_delta, std_delta = batched_delta_hyp(all_features, n_tries=30, batch_size=1000)
print(f"Relative delta hyperbolicity: {mean_delta:.6f} +/- {std_delta:.6f}")

100%|██████████| 30/30 [01:41<00:00,  3.39s/it]

Relative delta hyperbolicity: 0.385328 +/- 0.020822
